# Lab 9C: Variational Autoencoders — Can We Sample From the Latent Space?

**Before you start: Runtime → Change runtime type → GPU.**

This is the third of four notebooks. In 9B, the latent space of the autoencoder had gaps, an
arbitrary scale and an irregular shape, so we could not use it to **generate** new digits. The
variational autoencoder (VAE) fixes this. You have the VAE code from the lecture notes (Code 3); here we
take it apart, test each piece, and measure whether it does what it promises.

| Part | You will… |
|---|---|
| 0. A judge for generated images | set up a way to *measure* the quality of generated images |
| 1. The baseline: generating with an autoencoder | try to generate digits with a normal autoencoder |
| 2. The VAE, piece by piece | implement and test the reparameterisation trick and the KL term, train a VAE, and compare its latent space with the autoencoder's |
| 3. The β dial | change the weight of the KL term, and measure the trade-off |
| 4. Exam-style questions | check your understanding |

## Setup

In [ ]:
import copy
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ---- Data: 10% of the MNIST training set, and 2,000 test images (used as a validation set) ----
FRACTION = 0.1
mnist_train = torchvision.datasets.MNIST('./data', train=True, download=True)
mnist_test = torchvision.datasets.MNIST('./data', train=False, download=True)
idx = torch.randperm(len(mnist_train), generator=torch.Generator().manual_seed(0))[:int(FRACTION * len(mnist_train))]

# The whole (small) dataset fits on the GPU, so we skip the DataLoader and index tensors directly.
X_train = (mnist_train.data[idx].float() / 255).unsqueeze(1).to(device)    # (6000, 1, 28, 28), values in [0, 1]
y_train = mnist_train.targets[idx].to(device)                               # labels: used ONLY for colouring plots
X_test = (mnist_test.data[:2000].float() / 255).unsqueeze(1).to(device)    # (2000, 1, 28, 28)
y_test = mnist_test.targets[:2000].to(device)
print('X_train:', tuple(X_train.shape), ' X_test:', tuple(X_test.shape))


def batches(n, batch_size=64, shuffle=True):
    # Mini-batches of indices: the same job a DataLoader(shuffle=True) does for you.
    order = torch.randperm(n, device=device) if shuffle else torch.arange(n, device=device)
    return order.split(batch_size)


def recon_loss(x_hat, x):
    # J = (1/n) * sum_i ||x_hat_i - x_i||^2 : squared error summed over pixels, averaged over the batch
    return ((x_hat - x) ** 2).flatten(1).sum(1).mean()


def show_images(x, nrow=10, title=None, ax=None, figsize=None):
    # Show a batch of (N, 1, 28, 28) images as one grid.
    x = x.detach().float().cpu().clamp(0, 1)
    grid = torchvision.utils.make_grid(x, nrow=nrow, padding=1, pad_value=1)[0]
    if ax is None:
        plt.figure(figsize=figsize or (nrow * 0.6, (len(x) + nrow - 1) // nrow * 0.6))
        ax = plt.gca()
    ax.imshow(grid, cmap='gray', vmin=0, vmax=1)
    ax.axis('off')
    if title:
        ax.set_title(title)


def plot_latent(z, labels, ax=None, title=None, lim=None):
    # Scatter plot of 2D latent codes, coloured by the true digit label (labels are never used in training).
    z, labels = z.detach().cpu().numpy(), labels.cpu().numpy()
    ax = ax or plt.gca()
    sc = ax.scatter(z[:, 0], z[:, 1], c=labels, cmap='tab10', s=4, vmin=-0.5, vmax=9.5)
    plt.colorbar(sc, ax=ax, ticks=range(10))
    if lim is not None:
        ax.set_xlim(lim[0]); ax.set_ylim(lim[1])
    ax.set_xlabel('z[0]'); ax.set_ylabel('z[1]')
    if title:
        ax.set_title(title)


@torch.no_grad()
def plot_latent_grid(decode, xlim, ylim, n=20, ax=None, title=None):
    # Decode an n x n grid of latent vectors and show each decoded image at its position in latent space.
    gx = torch.linspace(xlim[0], xlim[1], n)
    gy = torch.linspace(ylim[1], ylim[0], n)            # top row = largest z[1], as in a scatter plot
    z = torch.stack(torch.meshgrid(gy, gx, indexing='ij')[::-1], -1).view(-1, 2).to(device)
    imgs = decode(z).view(n, n, 28, 28).cpu()
    canvas = imgs.permute(0, 2, 1, 3).reshape(n * 28, n * 28)
    ax = ax or plt.gca()
    ax.imshow(canvas, cmap='gray', extent=[xlim[0], xlim[1], ylim[0], ylim[1]], aspect='auto', vmin=0, vmax=1)
    ax.set_xlabel('z[0]'); ax.set_ylabel('z[1]')
    if title:
        ax.set_title(title)


# ---- The "digit judge": a small CNN classifier trained on the FULL MNIST training set ----
# We use it only to SCORE generated images (it is never used to train a generative model).
class Judge(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 14x14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 7x7
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        return self.net(x)


def get_judge(path='judge.pt'):
    judge = Judge().to(device)
    try:
        judge.load_state_dict(torch.load(path, map_location=device))
        print('Loaded the judge from', path)
    except FileNotFoundError:
        set_seed(0)
        Xf = (mnist_train.data.float() / 255).unsqueeze(1).to(device)
        yf = mnist_train.targets.to(device)
        opt = torch.optim.Adam(judge.parameters(), 1e-3)
        for epoch in range(2):
            for b in batches(len(Xf), 128):
                loss = F.cross_entropy(judge(Xf[b]), yf[b])
                opt.zero_grad(); loss.backward(); opt.step()
        torch.save(judge.state_dict(), path)
    judge.eval()
    with torch.no_grad():
        acc = (judge(X_test).argmax(1) == y_test).float().mean().item()
    print(f'Judge accuracy on real test digits: {acc:.1%}')
    return judge


@torch.no_grad()
def judge_scores(images, judge):
    # confidence: mean of the judge's top softmax probability  ("does it look like *some* digit?")
    # coverage:   entropy of the predicted-class histogram / log(10)  (1.0 = all ten digits equally often,
    #             0.0 = always the same digit, i.e. no variety)
    p = F.softmax(judge(images.clamp(0, 1)), dim=1)
    confidence = p.max(1).values.mean().item()
    counts = torch.bincount(p.argmax(1), minlength=10).float()
    freq = counts / counts.sum()
    coverage = -(freq[freq > 0] * freq[freq > 0].log()).sum().item() / math.log(10)
    return {'confidence': round(confidence, 3), 'coverage': round(coverage, 3)}


def sharpness(x):
    # mean absolute difference between neighbouring pixels (see 9B): low = blurry
    dx = (x[..., :, 1:] - x[..., :, :-1]).abs().mean()
    dy = (x[..., 1:, :] - x[..., :-1, :]).abs().mean()
    return (dx + dy).item()

---
# Part 0 — A judge for generated images

How do we decide whether one generative model is better than another? Looking at samples is
important, but it is subjective, and it is easy to pick the nice ones. Research papers therefore use
**pretrained classifiers** to score generated images (the best-known scores are the *Inception
Score* and the *FID*). We use a simple version of the same idea: a small CNN trained on the **full**
MNIST training set (our "judge"), and two scores computed from its predictions on a batch of
generated images:

* **confidence**: the average of the judge's highest softmax probability. High if the images look
  like *some* digit; low for images the judge cannot classify.
* **coverage**: how evenly the judge's predicted classes are spread over the ten digits (the entropy
  of the class histogram, divided by $\log 10$). 1.0 means all digits appear equally often; 0.0
  means every image is the same digit. A good generator should produce **all** digits.

The judge is only used to *score* images; it is never used to train the generative models. Training
it takes about 20 seconds on a GPU.

In [ ]:
judge = get_judge()
print('real test digits:      ', judge_scores(X_test, judge))
print('random noise images:   ', judge_scores(torch.rand(2000, 1, 28, 28, device=device), judge))
print('blurred test digits:   ', judge_scores(F.avg_pool2d(X_test, 3, 1, 1), judge))
print('the same digit 2,000×: ', judge_scores(X_test[:1].repeat(2000, 1, 1, 1), judge))
print(f'sharpness: real {sharpness(X_test):.3f}   blurred {sharpness(F.avg_pool2d(X_test, 3, 1, 1)):.3f}')

### Questions 0

1. Which property of a generator does each score measure? Which score detects a generator that
   produces only one kind of digit?
2. The judge is still confident on blurred digits. What does that tell you about what the judge
   does **not** measure? (That's why we also measure sharpness.)
3. Can a generator get a perfect score with images that a human would find bad? How?

---
# Part 1 — The baseline: generating with an autoencoder

This is the convolutional autoencoder from 9B (with the shapes fixed), with a 2D latent space.

In [ ]:
class ConvAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten(), nn.Linear(64 * 7 * 7, latent_dim))
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64 * 7 * 7), nn.ReLU(), nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid())

    def encode(self, x):  return self.encoder(x)
    def decode(self, z):  return self.decoder(z)
    def forward(self, x): return self.decode(self.encode(x))


set_seed(0)
ae = ConvAE(2).to(device)
opt = torch.optim.Adam(ae.parameters(), 1e-3)
for epoch in range(30):
    for b in batches(len(X_train), 64):
        loss = recon_loss(ae(X_train[b]), X_train[b])
        opt.zero_grad(); loss.backward(); opt.step()
ae.eval()
with torch.no_grad():
    z_ae = ae.encode(X_test)
    print(f'AE test reconstruction loss: {recon_loss(ae(X_test), X_test).item():.1f}')
print('mean of the latent codes:', z_ae.mean(0).cpu().numpy().round(2), '  std:', z_ae.std(0).cpu().numpy().round(2))

To generate, we need a distribution to sample latent vectors from. We try two options:

1. the standard normal distribution $\mathcal{N}(\mathbf{0}, \mathbf{I})$, and
2. a Gaussian with the **same mean and covariance as the latent codes** of the training images
   (a fairer choice: it is at least in the right place, with the right scale).

✍️ **Prediction:** which option will work better? Will either of them produce all ten digits?

In [ ]:
@torch.no_grad()
def generate_and_score(decode, z, name, show=True):
    x = decode(z)
    scores = judge_scores(x, judge)
    scores['sharpness'] = round(sharpness(x) / sharpness(X_test), 2)
    print(f'{name:40s}', scores)
    if show:
        show_images(x[:30], nrow=15, title=name, figsize=(9, 1.5))
        plt.show()
    return scores


results = {}
set_seed(1)
results['AE, z ~ N(0, I)'] = generate_and_score(ae.decode, torch.randn(2000, 2, device=device), 'AE, z ~ N(0, I)')

with torch.no_grad():
    z_tr = ae.encode(X_train)
fitted = torch.distributions.MultivariateNormal(z_tr.mean(0), torch.cov(z_tr.T))
results['AE, z ~ Gaussian fitted to codes'] = generate_and_score(ae.decode, fitted.sample((2000,)), 'AE, z ~ Gaussian fitted to codes')

---
# Part 2 — The VAE, piece by piece

The VAE changes two things compared with the autoencoder (lecture, Section 4):

1. The encoder outputs a **distribution** for each image, a Gaussian $\mathcal{N}(\boldsymbol\mu, \boldsymbol\sigma^2)$,
   instead of a single point. The decoder gets a **sample** $\mathbf{z}$ from it.
2. The loss gets a second term, the **KL divergence**, that pulls all these distributions towards
   the standard normal distribution $\mathcal{N}(\mathbf{0}, \mathbf{I})$.

The architecture is the one from the lecture notes. Note that the encoder outputs $\log\sigma^2$
(`log_var`), not $\sigma$: a linear layer can output any real number, but a standard deviation must be
positive. With $\sigma = \exp(\tfrac{1}{2}\log\sigma^2)$, every output of the layer corresponds to a
valid $\sigma > 0$.

### The reparameterisation trick (explained)

During training, we want to backpropagate the loss through the sampled $\mathbf{z}$ to the encoder, to
learn $\boldsymbol\mu$ and $\boldsymbol\sigma$. But "draw a random sample from
$\mathcal{N}(\boldsymbol\mu, \boldsymbol\sigma^2)$" is not an operation we can differentiate: how would the
sample change if $\mu$ changed slightly? There is no formula for that.

The trick is to draw the randomness **separately**, from a distribution that does not depend on any
parameter, and then *transform* it:

$$\boldsymbol\varepsilon \sim \mathcal{N}(\mathbf{0}, \mathbf{I}), \qquad \mathbf{z} = \boldsymbol\mu + \boldsymbol\sigma \odot \boldsymbol\varepsilon .$$

This gives exactly the same distribution for $\mathbf{z}$ (shifting a standard normal by $\mu$ and
scaling it by $\sigma$ gives $\mathcal{N}(\mu, \sigma^2)$). But now, **once $\boldsymbol\varepsilon$ has been
drawn**, $\mathbf{z}$ is an ordinary differentiable function of $\boldsymbol\mu$ and $\boldsymbol\sigma$,
with $\partial z_j / \partial \mu_j = 1$ and $\partial z_j / \partial \sigma_j = \varepsilon_j$. In the
computational graph (Lecture 3), $\boldsymbol\varepsilon$ is just another input, like the image, and the
chain rule takes care of the rest:

```
x ─► encoder ─► μ ────────────┐
              └► log σ² ─► σ ─┴─(μ + σ·ε)─► z ─► decoder ─► x̂ ─► loss
                     ε ~ N(0, I) ┘   (no gradient needed for ε)
```

### 🛠️ TASK 2a: Implement `reparameterise`

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Flatten())
        self.fc_mu = nn.Linear(64 * 7 * 7, latent_dim)          # mean
        self.fc_log_var = nn.Linear(64 * 7 * 7, latent_dim)     # log(sigma^2)
        self.decoder = nn.Sequential(                           # identical to the AE's decoder
            nn.Linear(latent_dim, 64 * 7 * 7), nn.ReLU(), nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1), nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 3, stride=2, padding=1, output_padding=1), nn.Sigmoid())

    def encode(self, x):
        h = self.encoder_conv(x)
        return self.fc_mu(h), self.fc_log_var(h)

    def reparameterise(self, mu, log_var):
        std = ???                     # sigma from log(sigma^2)
        eps = ???                     # a standard normal sample with the same shape as std
        return ???                    # z = mu + sigma * eps

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparameterise(mu, log_var)
        return self.decode(z), mu, log_var

In [ ]:
# Tests for Task 2a
_vae = VAE(2)
_mu = torch.tensor([[1.0, -2.0]]).repeat(100_000, 1)
_lv = torch.log(torch.tensor([[0.25, 4.0]])).repeat(100_000, 1)        # sigma = 0.5 and 2.0
_z = _vae.reparameterise(_mu, _lv)
assert _z.shape == _mu.shape
print('sample mean:', _z.mean(0).numpy().round(2), ' (should be [ 1. -2.])')
print('sample std: ', _z.std(0).numpy().round(2), ' (should be [0.5 2. ])')
assert torch.allclose(_z.mean(0), torch.tensor([1.0, -2.0]), atol=0.03), 'the mean of z should be mu'
assert torch.allclose(_z.std(0), torch.tensor([0.5, 2.0]), atol=0.03), 'the std of z should be exp(0.5 * log_var)'

# Gradients must flow back to mu and log_var: dz/dmu = 1, dz/dlog_var = 0.5 * sigma * eps
_mu = torch.zeros(5, 2, requires_grad=True)
_lv = torch.zeros(5, 2, requires_grad=True)
_vae.reparameterise(_mu, _lv).sum().backward()
assert _mu.grad is not None and torch.allclose(_mu.grad, torch.ones(5, 2)), 'dz/dmu should be 1'
assert _lv.grad is not None and _lv.grad.abs().sum() > 0, 'no gradient reaches log_var'
print('✅ Task 2a passed')

### The KL term (explained)

The **Kullback–Leibler (KL) divergence** $D_\text{KL}(q \,\|\, p)$ measures how different a
distribution $q$ is from a reference distribution $p$. It is never negative, and it is zero only if
$q = p$. (Lecture 13 explains where it comes from, using entropy and cross-entropy.) Its definition is
an **expected value**:

$$D_\text{KL}(q \,\|\, p) = \mathbb{E}_{\mathbf{z} \sim q}\big[\log q(\mathbf{z}) - \log p(\mathbf{z})\big],$$

"on average over samples from $q$, how much more likely is the sample under $q$ than under $p$?".

In the VAE, $q = \mathcal{N}(\boldsymbol\mu, \boldsymbol\sigma^2)$ is the distribution the encoder outputs
for one image, and $p = \mathcal{N}(\mathbf{0}, \mathbf{I})$. For two Gaussians, the expected value can
be worked out exactly (the formula in the lecture):

$$D_\text{KL}\big(\mathcal{N}(\boldsymbol\mu, \boldsymbol\sigma^2) \,\|\, \mathcal{N}(\mathbf{0}, \mathbf{I})\big) = \frac{1}{2}\sum_j \left(\mu_j^2 + \sigma_j^2 - \log\sigma_j^2 - 1\right).$$

Look at the terms: $\mu_j^2$ is smallest when $\mu_j = 0$ (it pulls the code towards the origin), and
$\sigma_j^2 - \log\sigma_j^2 - 1$ is smallest when $\sigma_j = 1$ (it stops the encoder from shrinking the
distribution to a single point, which would turn the VAE back into a normal autoencoder).

### 🛠️ TASK 2b: Implement the KL term, and check it against its definition

1. Implement the closed-form formula. Sum over the latent dimensions, and average over the images in
   the batch (just like `recon_loss`).
2. Check it. We can also *estimate* an expected value by averaging over many samples (a
   **Monte-Carlo** estimate): draw $\mathbf{z}^{(1)}, \dots, \mathbf{z}^{(S)}$ from $q$ and compute
   $\frac{1}{S}\sum_s \big[\log q(\mathbf{z}^{(s)}) - \log p(\mathbf{z}^{(s)})\big]$. If the closed-form
   formula is implemented correctly, the two must agree (up to sampling noise). The function
   `log_normal` (the log density of a Gaussian) is given. Fill in the Monte-Carlo estimate.

In training we use the closed form: it is exact, cheap, and adds no extra noise to the gradients. The
Monte-Carlo estimate is what you would use for distributions where no closed form exists, and here it
gives us an independent test of the formula.

In [ ]:
def kl_divergence(mu, log_var):
    # closed form, summed over latent dimensions, averaged over the batch
    return ???


def log_normal(z, mu, log_var):
    # log N(z; mu, sigma^2) for a diagonal Gaussian, summed over the latent dimensions
    return (-0.5 * (math.log(2 * math.pi) + log_var + (z - mu) ** 2 / log_var.exp())).sum(-1)


# A few made-up encoder outputs (3 images, 2 latent dimensions)
mu_test = torch.tensor([[0.0, 0.0], [1.0, -0.5], [2.0, 0.3]])
log_var_test = torch.tensor([[0.0, 0.0], [-1.0, 0.5], [-2.0, -0.2]])

S = 200_000
z = mu_test + torch.exp(0.5 * log_var_test) * torch.randn(S, 3, 2)      # S samples from q for each image: (S, 3, 2)
log_q = ???                                                             # log q(z): (S, 3)
log_p = ???                                                             # log p(z) with p = N(0, I): (S, 3)
kl_mc = ???                                                             # Monte-Carlo estimate, averaged over the 3 images

print(f'closed form: {kl_divergence(mu_test, log_var_test).item():.4f}    Monte Carlo: {kl_mc.item():.4f}')

In [ ]:
# Tests for Task 2b
assert abs(kl_divergence(torch.zeros(4, 2), torch.zeros(4, 2)).item()) < 1e-6, 'KL(N(0,1) || N(0,1)) must be 0'
assert abs(kl_divergence(torch.ones(4, 2), torch.zeros(4, 2)).item() - 1.0) < 1e-5, 'for mu = 1, sigma = 1 in 2 dims, KL = 2 * 0.5 = 1'
assert abs(kl_divergence(mu_test, log_var_test).item() - kl_mc.item()) < 0.02, 'closed form and Monte-Carlo estimate disagree'
print('✅ Task 2b passed')

### The VAE loss, and training

As in the lecture notes, the loss is the reconstruction loss plus $\beta$ times the KL term. We keep
track of both terms separately: in a loss with two terms, the total alone can hide what is going on
(one term may improve while the other gets worse).

In [ ]:
def vae_loss(x_hat, x, mu, log_var, beta=1.0):
    recon = recon_loss(x_hat, x)
    kl = kl_divergence(mu, log_var)
    return recon + beta * kl, recon, kl


def train_vae(beta=1.0, latent_dim=2, epochs=30, seed=0, verbose=True):
    set_seed(seed)
    model = VAE(latent_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), 1e-3)
    history = {'recon': [], 'kl': []}
    for epoch in range(epochs):
        model.train()
        for b in batches(len(X_train), 64):
            x_hat, mu, log_var = model(X_train[b])
            loss, recon, kl = vae_loss(x_hat, X_train[b], mu, log_var, beta)
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            x_hat, mu, log_var = model(X_test)
            _, recon, kl = vae_loss(x_hat, X_test, mu, log_var, beta)
        history['recon'].append(recon.item()); history['kl'].append(kl.item())
        if verbose and (epoch + 1) % 10 == 0:
            print(f'epoch {epoch + 1:3d}   test recon {recon.item():6.1f}   test KL {kl.item():5.2f}')
    return model, history


t0 = time.time()
vae, hist = train_vae(beta=1.0)
print(f'Training time: {time.time() - t0:.0f} s')

fig, axs = plt.subplots(1, 2, figsize=(11, 3))
axs[0].plot(hist['recon']); axs[0].set_title('reconstruction loss (test)'); axs[0].set_xlabel('epoch')
axs[1].plot(hist['kl']); axs[1].set_title('KL term (test)'); axs[1].set_xlabel('epoch')
plt.tight_layout(); plt.show()

### Compare the latent spaces

Top row: the autoencoder from Part 1. Bottom row: the VAE. Left: where the encoder puts the test
images (for the VAE we plot the means $\boldsymbol\mu$). Right: the latent map, decoded over the region
$[-3, 3]^2$, which contains 99% of the probability mass of $\mathcal{N}(\mathbf{0}, \mathbf{I})$ in each
direction. For the autoencoder, we use the region of its codes as in 9B.

✍️ **Prediction:** how will the VAE's two plots differ from the autoencoder's?

In [ ]:
with torch.no_grad():
    mu_vae, _ = vae.encode(X_test)
print('AE codes:  mean', z_ae.mean(0).cpu().numpy().round(2), '  std', z_ae.std(0).cpu().numpy().round(2))
print('VAE means: mean', mu_vae.mean(0).cpu().numpy().round(2), '  std', mu_vae.std(0).cpu().numpy().round(2))
ae_lim = [(torch.quantile(z_ae[:, i], 0.01).item(), torch.quantile(z_ae[:, i], 0.99).item()) for i in range(2)]

fig, axs = plt.subplots(2, 2, figsize=(16, 14))
plot_latent(z_ae, y_test, ax=axs[0, 0], lim=ae_lim, title='AE: latent codes')
plot_latent_grid(ae.decode, *ae_lim, ax=axs[0, 1], title='AE: latent map')
plot_latent(mu_vae, y_test, ax=axs[1, 0], lim=((-3, 3), (-3, 3)), title='VAE: latent means μ')
plot_latent_grid(vae.decode, (-3, 3), (-3, 3), ax=axs[1, 1], title='VAE: latent map')
plt.tight_layout(); plt.show()

### Generating new digits

For the VAE, generation is: sample $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and decode it.
The encoder is not needed.

In [ ]:
set_seed(1)
results['VAE (β = 1), z ~ N(0, I)'] = generate_and_score(vae.decode, torch.randn(2000, 2, device=device), 'VAE (β = 1), z ~ N(0, I)')
print()
for name, r in results.items():
    print(f'{name:40s}', r)

### Questions 2

1. Describe the differences between the AE's and the VAE's latent codes and latent maps. Which plot
   shows you that the gaps are gone?
2. Which loss term is responsible for each difference you see? Explain in terms of what the
   decoder sees during training (the role of the **sampling**) and of what the **KL term** does.
3. Compare the scores. Did the VAE generate better digits than the AE? Use both confidence and
   coverage, and say which AE sampling method you compare with.
4. **Mechanism vs. outcome:** the KL term clearly did its job on the latent space. Did that translate
   into better samples than the AE with the *fitted* Gaussian? If not, what *does* the VAE give you that
   the AE doesn't?
5. Look at the sharpness of the samples. What is the weakness of the VAE samples, and where does it
   come from?
6. Look at the loss curves. Did the KL term go down during training? Explain what you see.

### Interpolation in the VAE

Let's repeat the 7 → 9 interpolation from 9B with both models, using the means $\boldsymbol\mu$ for the VAE.

In [ ]:
x7, x9 = X_test[y_test == 7][:1], X_test[y_test == 9][:1]
alphas = torch.linspace(0, 1, 10, device=device).view(-1, 1)
with torch.no_grad():
    za7, za9 = ae.encode(x7), ae.encode(x9)
    zv7, zv9 = vae.encode(x7)[0], vae.encode(x9)[0]
    path_ae = ae.decode((1 - alphas) * za7 + alphas * za9)
    path_vae = vae.decode((1 - alphas) * zv7 + alphas * zv9)
show_images(torch.cat([path_ae, path_vae]), nrow=10, title='7 → 9:   AE (top)   ·   VAE (bottom)', figsize=(8, 1.8))

---
# Part 3 — The β dial

The two terms of the VAE loss pull in opposite directions: the reconstruction term wants codes that
are far apart and precise (easy to decode), and the KL term wants all codes to look like
$\mathcal{N}(\mathbf{0}, \mathbf{I})$. The weight $\beta$ sets the balance. We train the same VAE with four values
of $\beta$, changing nothing else.

✍️ **Prediction** (write it down before you run the cell). As $\beta$ increases:
* **Mechanism:** what happens to the KL term?
* **Outcome:** what happens to the reconstruction loss, and to the quality of generated samples
  (confidence, coverage, sharpness)? Is there a best $\beta$?

In [ ]:
betas = [0.1, 1.0, 5.0, 25.0]
sweep = {}
t0 = time.time()
for beta in betas:
    model, h = train_vae(beta=beta, epochs=20, verbose=False)
    set_seed(1)
    scores = generate_and_score(model.decode, torch.randn(2000, 2, device=device), f'β = {beta}', show=False)
    sweep[beta] = dict(model=model, recon=h['recon'][-1], kl=h['kl'][-1], **scores)
print(f'Training time: {time.time() - t0:.0f} s\n')

print(f'{"β":>6s} {"recon":>8s} {"KL":>7s} {"confidence":>11s} {"coverage":>9s} {"sharpness":>10s}')
for beta, r in sweep.items():
    print(f'{beta:6.1f} {r["recon"]:8.1f} {r["kl"]:7.2f} {r["confidence"]:11.3f} {r["coverage"]:9.3f} {r["sharpness"]:10.2f}')

fig, axs = plt.subplots(2, len(betas), figsize=(5 * len(betas), 10))
for j, (beta, r) in enumerate(sweep.items()):
    with torch.no_grad():
        mu, _ = r['model'].encode(X_test)
    plot_latent(mu, y_test, ax=axs[0, j], lim=((-3, 3), (-3, 3)), title=f'β = {beta}: latent means')
    plot_latent_grid(r['model'].decode, (-3, 3), (-3, 3), n=12, ax=axs[1, j], title=f'β = {beta}: latent map')
plt.tight_layout(); plt.show()

### Questions 3

1. **Mechanism:** describe what happens to the KL term and to the latent codes as $\beta$ grows.
2. **Outcome:** what happens to the reconstruction loss and to the sample scores? Which $\beta$ gives the
   best samples according to the judge, and is it also the one with the best reconstructions?
3. With a very small $\beta$, the VAE behaves more and more like a normal autoencoder. Which of the plots
   and numbers show this?
4. With a very large $\beta$, the codes of all images end up on top of each other at the origin. What does
   the decoder then produce, and why? (This is called **posterior collapse**: the latent code no longer
   carries information about the input.)
5. The best $\beta$ depends on what you want the model for. Which $\beta$ would you choose for
   (a) generating new samples, (b) compressing images, (c) a smooth latent space for interpolation?

---
# Part 4 — Exam-style questions

**Q1.** Why does a VAE's encoder output $\log\sigma^2$ rather than $\sigma$?

* (a) Because $\log\sigma^2$ is always positive.
* (b) Because a linear layer can output any real number, and $\exp(\tfrac{1}{2}\log\sigma^2)$ turns every such number into a valid, positive $\sigma$.
* (c) Because the KL term cannot be computed from $\sigma$.
* (d) Because it makes the decoder faster.

**Q2.** What does the reparameterisation trick $\mathbf{z} = \boldsymbol\mu + \boldsymbol\sigma \odot \boldsymbol\varepsilon$, $\boldsymbol\varepsilon \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$, make possible?

* (a) Sampling $\mathbf{z}$ without an encoder.
* (b) Backpropagating through the sampling step to $\boldsymbol\mu$ and $\boldsymbol\sigma$.
* (c) Computing the KL term in closed form.
* (d) Training without a reconstruction loss.

**Q3.** What happens if the KL term is removed from the VAE loss (β = 0)?

* (a) The model can no longer reconstruct its inputs.
* (b) The latent codes are no longer pulled towards $\mathcal{N}(\mathbf{0}, \mathbf{I})$, so the latent space can again have gaps and an arbitrary scale, and sampling from $\mathcal{N}(\mathbf{0}, \mathbf{I})$ no longer works well.
* (c) The samples become sharper and more varied.
* (d) Nothing; the KL term only speeds up training.

**Q4.** How do you generate a new image with a trained VAE?

* (a) Encode a random training image and decode its $\boldsymbol\mu$.
* (b) Sample $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and pass it through the decoder.
* (c) Sample $\boldsymbol\varepsilon \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ and pass it through the encoder.
* (d) Average the reconstructions of all training images.

---
**Write down your numbers** (the scores of the AE and of the VAE with β = 1): you will compare them with
the GAN in 9D.

**Next: notebook 9D**, where we get rid of the reconstruction loss altogether: generative adversarial
networks.